In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv")

In [4]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


# Use only the following columns:

'engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg'

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  str    
 2   fuel_type            10000 non-null  str    
 3   drivetrain           10000 non-null  str    
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), str(3)
memory usage: 859.5 KB


In [6]:
prep_df = df[["engine_displacement", "horsepower", "vehicle_weight", "model_year", "fuel_efficiency_mpg"]]

In [7]:
prep_df.info(0)

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  10000 non-null  int64  
 1   horsepower           9123 non-null   float64
 2   vehicle_weight       10000 non-null  int64  
 3   model_year           10000 non-null  int64  
 4   fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 390.8 KB


In [8]:
prep_df.isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [9]:
prep_df["horsepower"].describe()

count    9123.000000
mean      254.446016
std        22.844613
min       171.000000
25%       239.000000
50%       254.000000
75%       270.000000
max       334.000000
Name: horsepower, dtype: float64

In [10]:
prep_df["horsepower"].median()

np.float64(254.0)

In [11]:
# dataset split
def split_df(seed_value = 42):
    np.random.seed(seed_value) # to make shuffling reproducible

    n = len(prep_df)

    n_val = int(0.2 * n)
    n_test = int(0.2 * n)
    n_train = n - (n_val + n_test)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_shuffled = prep_df.iloc[idx]

    df_train = df_shuffled.iloc[:n_train].copy()
    df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
    df_test = df_shuffled.iloc[n_train+n_val:].copy()
    return df_train, df_val, df_test

In [12]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0]) #this is where regularization starts
    XTX = XTX + reg #regularization is applied here

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

In [13]:
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [14]:
# Question 3
df_train, df_val, df_test = split_df(42)
#resetting index
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

mean = df_train['horsepower'].mean()

y_train_orig = df_train.fuel_efficiency_mpg.values
y_val_orig = df_val.fuel_efficiency_mpg.values
y_test_orig = df_test.fuel_efficiency_mpg.values

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

print(f"mean = {mean}")
missing_value = [0, mean]
for n in missing_value:
    train = df_train.copy()
    val = df_val.copy()

    train = train.fillna(n)
    val = val.fillna(n)

    X_train = train.values
    X_val = val.values

    w_0, w = train_linear_regression_reg(X_train, y_train)

    y_pred = w_0 + X_val.dot(w)

    score = round(rmse(y_val, y_pred), 3)

    print(f"Missing value filled with {n}: RMSE = {score}")

mean = 254.46338337605272
Missing value filled with 0: RMSE = 0.071
Missing value filled with 254.46338337605272: RMSE = 0.071


In [36]:
# Question 4 
df_train, df_val, df_test = split_df(42)
#resetting index
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train_orig = df_train.fuel_efficiency_mpg.values
y_val_orig = df_val.fuel_efficiency_mpg.values
y_test_orig = df_test.fuel_efficiency_mpg.values

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

train = df_train.drop(columns=['fuel_efficiency_mpg']).copy()
val = df_val.drop(columns=['fuel_efficiency_mpg']).copy()

train = train.fillna(0)
val = val.fillna(0)

X_train = train.values
X_val = val.values


lowest_r = float('inf')
lowest_rmse = float('inf')


for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w_0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w_0 + X_val.dot(w)
    cal_rmse = rmse(y_val, y_pred)
    if(cal_rmse < lowest_rmse):
        lowest_rmse = cal_rmse
        lowest_r = r
    print('%6s' %r, rmse(y_val, y_pred))
print("Printing lowest")
print(f"lowest r = {lowest_r}, lowest rmse = {round(lowest_rmse,4)}")

     0 0.07148851150877135
  0.01 0.0714932588417139
   0.1 0.07165023237544382
     1 0.07274035044837335
     5 0.07327911622523903
    10 0.07337041231449234
   100 0.07345871819237115
Printing lowest
lowest r = 0, lowest rmse = 0.0715


In [32]:
train

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,1900,239.0,3790,2015,32.5
1,2000,224.0,4380,1992,29.1
2,2330,286.0,4090,2022,34.1
3,2300,0.0,4150,1997,30.6
4,2340,269.0,4400,2001,30.7
...,...,...,...,...,...
5995,2280,242.0,4590,2018,29.8
5996,2300,0.0,4240,1984,25.1
5997,2240,255.0,4190,1989,27.7
5998,2430,254.0,4280,2020,31.6


In [38]:
# Question 5 without log1p
rmse_scores = []
seed_values = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

for seed in seed_values:
    df_train, df_val, df_test = split_df(seed)

    # Target
    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values

  
    # Features: remove target
    train = df_train.drop(columns=['fuel_efficiency_mpg']).copy()
    val = df_val.drop(columns=['fuel_efficiency_mpg']).copy()

    # Fill missing values with 0
    train = train.fillna(0)
    val = val.fillna(0)

    X_train = train.values
    X_val = val.values

    # No regularization
    w_0, w = train_linear_regression_reg(X_train, y_train)

    y_pred = w_0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    rmse_scores.append(score)

print(rmse_scores)
print(round(np.std(rmse_scores), 3))


[np.float64(2.2392041430485716), np.float64(2.202112821083587), np.float64(2.163419778753105), np.float64(2.2043588768541773), np.float64(2.201880094336824), np.float64(2.2457851509077487), np.float64(2.269721207952116), np.float64(2.190794599936849), np.float64(2.2166112016917348), np.float64(2.2070365928200184)]
0.029


In [30]:
# Question 6

df_train, df_val, df_test = split_df(9)
#resetting index
df_train_val_merged = pd.concat([df_train, df_val], axis=0)
df_train = df_train_val_merged.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values


train = df_train.drop(columns=['fuel_efficiency_mpg']).copy()
test = df_test.drop(columns=['fuel_efficiency_mpg']).copy()

train = train.fillna(0)
test = test.fillna(0)

X_train = train.values
X_test = test.values

w_0, w = train_linear_regression_reg(X_train, y_train)
y_pred = w_0 + X_test.dot(w)
cal_rmse = round(rmse(y_test, y_pred),3)

print(cal_rmse)


2.236
